# Day 031 · 环境搭建:Anaconda 工作流 · 中国版
**Setup** · 阶段 P2 · Python 量化工具栈

> 今天是 P2 工程化阶段的第一节。前 29 天我们写过的所有 Python 代码,环境基本是「装个 Python 直接开搞」。这种做法在写一两个小脚本时没问题,做正经量化项目立刻翻车。今天讲清楚四件事 — 第一,为什么必须用虚拟环境而不是直接装包到系统 Python,讲一个让所有工程师都心有余悸的真实故事;第二,Anaconda 跟 Miniconda、pip 跟 poetry 这几个工具到底什么差别,量化场景里怎么选;第三,量化必装的 15 个包清单 + 国内镜像源配置,装好之后你的环境跟头部私募研究员的工作环境基本一致;第四,Jupyter Lab 跟 VS Code 各自适合什么场景、什么时候用哪个。最后给你一个标准的量化项目目录结构 + .gitignore 模板,直接抄走就行。学完这一节你的工作环境就升级到了「专业研究员」水平,后面 P2 P3 P4 所有项目都站在这个基础上。

---

### 关于「中国版」

本 notebook 是为**国内学员**优化的版本:
- 数据源用 **akshare**(国内可访问、零 VPN、免注册),取代了视频里的 yfinance
- 标的尽量保持原意:美股 ETF→A 股 ETF / 国际公司→A 股龙头
- 所讲的**概念和方法 100% 一致**,但**具体数字可能与视频里略有差异**(因为是不同时间窗 / 不同标的)
- 一般情况国内 `pip install akshare` 即可,无需 token / VPN

**课件生成日期:** 2026-05-20  ·  **建议学习时长:** 20 分钟

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有必需的 Python 包(含 `akshare`),缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续


In [1]:
# === 环境自检 + 自动安装(运行此单元格即可)===
import importlib, subprocess, sys, os

REQUIRED = ["akshare", "matplotlib", "numpy", "numpy_financial", "pandas", "platform", "scipy", "sklearn", "statsmodels", "warnings"]
PIP_NAME = {"sklearn":"scikit-learn","cv2":"opencv-python","PIL":"Pillow","bs4":"beautifulsoup4","yaml":"PyYAML"}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))
if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置 ===
import matplotlib, matplotlib.pyplot as plt, matplotlib.font_manager as fm
CJK = ["/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
       "C:/Windows/Fonts/msyh.ttc","C:/Windows/Fonts/simhei.ttf",
       "/System/Library/Fonts/PingFang.ttc","/System/Library/Fonts/STHeiti Medium.ttc"]
for p in CJK:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP","Microsoft YaHei","PingFang SC","SimHei","DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪")


✓ 所有 10 个必需库已就绪
✓ 中文字体已加载:NotoSansCJK-Regular.ttc
✓ 环境就绪


## 🔌 第二步:加载国内数据助手

下面这一格是**工具函数**(可以折叠,不需要修改)。它把 `yfinance` 风格的 ticker(如 `600519.SS`)自动路由到对应的 akshare 接口,提供 `get_close(ticker)` 和 `get_close_multi(tickers_dict)` 两个函数。

In [2]:
# === 国内数据源助手(akshare 后端,不需要 VPN)===
# 这一格是工具函数,可以折叠,不需要修改。
# 它把 yfinance 风格的 ticker(如 "600519.SS" / "0700.HK" / "AAPL" / "BTC-USD")
# 自动路由到对应的 akshare 接口,统一返回 yfinance 风格的 Close DataFrame。

import re
from datetime import datetime, timedelta
import pandas as pd
import akshare as ak

_TICKER_MAP = {
    "^GSPC": ("us_index_sina", ".INX"),
    "^DJI":  ("us_index_sina", ".DJI"),
    "^IXIC": ("us_index_sina", ".IXIC"),
    "GC=F":  ("foreign_futures", "GC"),
    "SI=F":  ("foreign_futures", "SI"),
    "CL=F":  ("foreign_futures", "CL"),
    "BTC-USD": ("crypto", "BTC"),
    "ETH-USD": ("crypto", "ETH"),
}

def _retry(fn, *args, _retries=4, _wait=1.5, **kwargs):
    """akshare 上游(东方财富/新浪/Binance)偶有 RemoteDisconnected / Timeout,自动重试 4 次。
    2026-05-11 加:用户跑 cn notebook 拉 002594.SZ 时上游断连 → 整节卡死。
    每次重试间隔 _wait 秒(指数退避 1x → 1.5x → 2.25x)。
    """
    import time as _t
    last_err = None
    wait = _wait
    for i in range(_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            name = type(e).__name__
            if i == _retries - 1:
                print(f"  ✗ {getattr(fn,'__name__',str(fn))} 第 {i+1} 次仍失败({name}),放弃")
                raise
            print(f"  ⚠ {getattr(fn,'__name__',str(fn))} 第 {i+1} 次失败({name}),{wait:.1f}s 后重试")
            _t.sleep(wait)
            wait *= 1.5

def _parse_period(period):
    end = datetime.today()
    m = re.match(r"^(\d+)\s*(y|mo|d|w)$", period.lower().strip())
    days = 365 * 3 if not m else int(m.group(1)) * {"y":365,"mo":30,"w":7,"d":1}[m.group(2)]
    return (end - timedelta(days=days+30)).strftime("%Y%m%d"), end.strftime("%Y%m%d")

def _classify(ticker):
    t = ticker.strip()
    if t in _TICKER_MAP: return _TICKER_MAP[t]
    if t.endswith((".SS",".SH",".SZ")):
        code = t.split(".")[0]
        if code.startswith(("51","159","58")) or code in ("510300","510500","510050","511010","513100"):
            return ("a_etf", code)
        if code in ("000300","000016","000905","000852","000001"):
            return ("a_index", code)
        return ("a_stock", code)
    if t.endswith(".HK"):
        return ("hk", t.split(".")[0].zfill(5))
    return ("us", t)

def _norm(df, dc, cc):
    out = df[[dc, cc]].copy()
    out[dc] = pd.to_datetime(out[dc])
    return out.set_index(dc).sort_index()[cc].astype(float).rename("Close")

def get_close(ticker, period="3y"):
    """返回某标的 Close 价格 series。后端 akshare,中国可访问。
    所有 ak.* 调用都过 _retry(4 次,指数退避)— 防东方财富/新浪上游瞬时断连。
    """
    start, end = _parse_period(period)
    kind, sym = _classify(ticker)
    if kind == "a_stock":
        return _norm(_retry(ak.stock_zh_a_hist, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "a_etf":
        return _norm(_retry(ak.fund_etf_hist_em, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "a_index":
        idx_map = {"000300":"sh000300","000016":"sh000016","000905":"sh000905","000852":"sh000852","000001":"sh000001"}
        s = _norm(_retry(ak.stock_zh_index_daily_em, symbol=idx_map.get(sym, f"sh{sym}")), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "hk":
        return _norm(_retry(ak.stock_hk_hist, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "us":
        s = _norm(_retry(ak.stock_us_daily, symbol=sym, adjust="qfq"), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "us_index_sina":
        s = _norm(_retry(ak.index_us_stock_sina, symbol=sym), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "foreign_futures":
        s = _norm(_retry(ak.futures_foreign_hist, symbol=sym), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "crypto":
        import requests as _rq
        def _binance():
            r = _rq.get("https://api.binance.com/api/v3/klines",
                        params={"symbol": f"{sym}USDT", "interval": "1d", "limit": 1000}, timeout=15)
            r.raise_for_status()
            return r.json()
        klines = _retry(_binance)
        df = pd.DataFrame(klines, columns=["open_time","open","high","low","close","volume",
                                            "close_time","qav","trades","tbb","tbq","ignore"])
        df["date"] = pd.to_datetime(df["open_time"], unit="ms")
        df["close"] = df["close"].astype(float)
        s = df.set_index("date").sort_index()["close"].rename("Close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    raise ValueError(f"unsupported ticker: {ticker}")

def get_close_multi(tickers, period="3y"):
    """批量取 Close,返回 DataFrame,列名是 tickers dict 的 key(中文名),按交集日期对齐。"""
    series = {name: get_close(t, period=period) for name, t in tickers.items()}
    return pd.concat(series, axis=1).sort_index()

print("✓ cn_data 助手已加载 — 用 get_close(ticker) / get_close_multi(tickers_dict) 拉数据")


✓ cn_data 助手已加载 — 用 get_close(ticker) / get_close_multi(tickers_dict) 拉数据


## 学习目标

- 理解为什么必须用虚拟环境 — 通过一个真实的「包冲突」事故,看清楚不隔离的代价
- 区分 Anaconda / Miniconda / pip / poetry 四个工具的定位,知道量化场景里怎么选
- 搭建一个标准的量化研究环境 — 一行命令装好 15 个核心包,配国内镜像源加速 10 倍
- 选对你的主战场 IDE — Jupyter Lab(研究 / 探索 / 出报告)+ VS Code(写库 / 调试 / 部署)双轨制
- 学会一套标准的量化项目目录结构 + .gitignore 模板,直接复用到所有后续项目

## 历史背景:一个未隔离环境引发的连锁事故 — 量化新手的噩梦

讲一个真实案例。某私募研究员小张,2022 年刚入职,直接用电脑自带的 Python 装包。一开始没问题,他装了 pandas、numpy、matplotlib,做了几个回测项目,跑得挺顺。

3 个月后,公司接了个新项目要做深度学习因子,他装了 tensorflow 2.10。装完那一刻没事,第2 天打开之前的回测项目,所有代码都跑不起来了。报错信息一行接一行,提示 numpy 版本不兼容。原因是 tensorflow 2.10 要求 numpy 1.23,而他之前的回测项目用的是 numpy 1.21,旧 pandas 也只支持 numpy 1.21。tensorflow 装的时候自动把 numpy 升级了,把整个旧项目环境砸了。

他想退回去,pip uninstall numpy 然后 pip install numpy==1.21,结果 tensorflow 又跑不起来了。这次他想分别处理,把回测项目跟深度学习项目放不同目录,但代码运行的还是同一个 Python,同一套包,根本不存在「分别」。

他卡在这里两天,改了一堆代码强行妥协,最后发现总有一些库怎么都协调不了。第3 天他去找 CTO,CTO 说一句话 — 你为什么不用 conda 虚拟环境?他当场愣住,因为他从来不知道虚拟环境是什么,以为「装 Python = 装一个全局环境」就是全部。

CTO 花了 20 分钟教他 conda create / conda activate,他当晚把所有项目分别建独立环境,问题立刻解决。回头看,这两天他白损失了 16 小时工作,差点搞砸两个项目交付。

这个故事的核心教训不是 conda 多牛 — 是「不隔离 = 不可持续」。任何写过 1 年以上 Python 的人都遇到过类似事故。虚拟环境不是高级功能,是基础设施。今天这一节我们就讲清楚怎么搭、为什么搭、怎么避免小张的痛苦。这一节学完你就不会再踩这个坑,后面 P2 P3 都顺顺利利。



## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. 为什么必须用虚拟环境 — 隔离的工程价值

虚拟环境的核心价值,可以用一个简单的类比说清楚 — 工作环境就像你的房间,装的包就像你买的家具。如果所有项目共用一个房间(系统 Python),家具会越堆越多,有的家具尺寸冲突放不下,有的家具材质冲突放在一起会出问题。你最后只能在一个房间里凑合,所有项目都将就。

虚拟环境就是给每个项目一个独立的房间。回测项目一个房间装回测要用的家具(pandas 1.5、numpy 1.21、matplotlib 3.5),机器学习项目另一个房间装机器学习要用的家具(pandas 2.0、numpy 1.23、tensorflow 2.10),互不打扰。

这个隔离带来三个具体好处。第一,版本冲突彻底解决。每个项目锁定它自己的包版本,旧项目永远跑得动,新项目用最新版本。第二,可复现性。把你的环境配置(requirements.txt 或 environment.yml)分享给别人,别人 5 分钟内能在自己电脑上复制一个一模一样的环境,跑出同样的结果。第三,清理方便。某个项目不要了直接删整个环境,系统没有任何残留。系统 Python 装多了根本删不干净。

业内规则非常简单 — 任何超过 100 行代码的项目都必须用虚拟环境。低于 100 行的小脚本也建议用,只是不强求。专业团队里,「不用虚拟环境」基本等于「不专业」,跟数据科学家说「我直接 pip install 到系统 Python」对方会立刻怀疑你的工程能力。

虚拟环境是工程化的第一道关。过了这一关你才算正式入门,接下来的 Git、测试、CI 都建立在这一关之上。


### 2. Anaconda / Miniconda / pip / poetry — 怎么选

Python 包管理工具有很多种,新手最容易迷茫的就是「这么多工具我该用哪个」。下面我把量化场景里最常用的四个工具拆开讲,你看完就知道怎么选。

Anaconda — 一站式数据科学发行版。安装包 500MB+,装好之后自带 Python + 200 多个常用包 + Jupyter + 一个图形界面工具 Anaconda Navigator。优点是「装好就能用」,新手友好。缺点是体积大、占硬盘、自带很多你用不上的包。建议没碰过的新手第一次装 Anaconda,熟悉了之后切换到 Miniconda。

Miniconda — 简化版 Anaconda。安装包只有 50MB,只装最基本的 Python + conda 工具,其他包按需安装。优点是干净,你装的每个包都是你需要的。缺点是新手不习惯命令行装包。建议用过 1 个月 Anaconda 之后切换到 Miniconda。

pip + venv — Python 官方原生方案。pip 是 Python 自带的包管理器,venv 是 Python 自带的虚拟环境工具。完全免费、官方支持、最轻量。缺点是 conda 能装的某些 C/C++ 编译的包(比如 TensorFlow GPU 版),pip 装起来要自己处理依赖,容易出错。建议系统 Python 用户用,或者 Docker 容器内用。

poetry — 现代 Python 项目管理工具。功能比 pip 强,自动锁定依赖版本、自动生成 lock 文件、内置发布工具。优点是工程化最完整、上手成本相对低。缺点是社区比 pip 小,某些库不支持。建议正经做开源 Python 库用,做量化研究不太需要。

量化场景推荐 — Miniconda 主战场 + 偶尔 pip 补缺。Miniconda 装好之后用 conda create 建虚拟环境、conda activate 切换、conda install 装 conda 仓库的包、pip install 装 conda 没有的包。这套组合是国内 90% 量化机构的标配。其余几个工具知道就行,没必要深挖。


### 3. 必装包清单 + 国内镜像源

下面是量化研究员的「最低限度装包清单」,15 个包,一行命令装完。

数据处理(5 个) — numpy 数值计算基础、pandas 数据表操作、scipy 科学计算、statsmodels 统计建模、scikit-learn 机器学习。这 5 个是任何数据相关的基础。

可视化(3 个) — matplotlib 静态绘图基础、seaborn 高级统计图、plotly 交互式图表。matplotlib 必装,其他两个按需。

数据接口(3 个) — yfinance 海外免费、akshare 国内免费零翻墙、tushare 国内专业(可选)。这 3 个是数据来源主力,国内用户 akshare 必装,海外用户 yfinance 必装。

量化专用(4 个) — quantstats 一键策略报告、empyrical 指标计算、ffn 指标可视化一体、pyfolio 风险归因。这 4 个是后面 D29 那种「策略报告六件套」的快捷工具。

安装命令 — 一行搞定。
conda create -n quant python=3.11 -y
conda activate quant
pip install numpy pandas scipy statsmodels scikit-learn matplotlib seaborn plotly yfinance akshare quantstats empyrical ffn pyfolio-reloaded jupyter

国内镜像源加速 — pip 默认从 PyPI 海外下载,在国内速度可能慢 10 倍。配置清华镜像源后变得飞快。
pip config set global.index-url https://pypi.tuna.tsinghua.edu.cn/simple

conda 仓库也要配国内镜像源。在 ~/.condarc 文件里加上清华源(具体配置网上搜「conda 清华源」)。配完之后 conda install 也飞快。

这套环境装好之后,你就有了一台「能干 90% 量化工作」的电脑。占硬盘 5GB 左右,装一次能用很久。


### 4. Jupyter Lab vs VS Code — 双轨制

IDE 选哪个是量化新手第二个迷茫点。我直接告诉你结论 — 不要选,两个都用。Jupyter Lab 跟 VS Code 各自适合不同场景,组合起来才是完整工作流。

Jupyter Lab — 研究 / 探索 / 出报告的最佳工具。它的核心特点是「单元格驱动」,每段代码可以单独跑、单独看输出。这种交互特别适合数据探索 — 拉数据看一眼、画图看一眼、改参数看一眼、再画图看一眼。你不知道结果长什么样,就需要这种反复试探的工作流。所有的「数据探索」「图表绘制」「策略报告」「教学 notebook」都用 Jupyter Lab。我们这门课所有 notebook 都是这套工具产物。

VS Code — 写库 / 调试 / 部署的最佳工具。它的核心特点是「文件驱动」,整个项目是一棵文件树,所有操作围绕「编辑 .py 文件」展开。这种工作流适合「写一个完整的功能模块」「打包成库」「持续运行的脚本」「调试复杂 bug」。一旦你的代码超过 500 行,Jupyter 就不好维护了,要切到 VS Code。

建议工作流 — 研究阶段用 Jupyter Lab(探索数据、试模型、画图);代码定型后把核心逻辑搬到 .py 文件,用 VS Code 写成模块 / 类 / 测试;部署生产时用 VS Code 调试、git 管理、CI 跑测试。两个工具来回切换,没必要二选一。

配置建议 — VS Code 装这几个插件:Python(官方)、Pylance(智能补全)、Jupyter(在 VS Code 里也能开 notebook)、GitLens(增强 Git)、autoDocstring(自动写文档)。这套配置后你的 VS Code 就够用了。Jupyter Lab 装好之后基本不用配置,直接用。

注意点 — 千 万不要用电脑自带的「记事本 / TextEdit」写代码,那是非常痛苦的体验,新手做完一个项目就放弃量化的也不少。专业的工具能让你的效率提升 5 倍。


### 5. 量化项目目录结构 + .gitignore 模板

一个量化项目的标准目录结构,直接抄就行。

my_quant_project/
├── README.md(项目说明)
├── requirements.txt(依赖清单)
├── environment.yml(conda 环境配置)
├── .gitignore(忽略文件配置)
├── data/(原始数据)
│   ├── raw/(下载下来的原始数据)
│   └── processed/(清洗后的数据)
├── notebooks/(研究探索)
│   ├── 01_data_exploration.ipynb
│   ├── 02_strategy_test.ipynb
│   └── 03_results_analysis.ipynb
├── src/(核心代码模块)
│   ├── data/(数据获取与清洗)
│   ├── strategy/(策略逻辑)
│   ├── backtest/(回测引擎)
│   └── utils/(工具函数)
├── tests/(单元测试)
├── reports/(策略报告输出)
│   ├── figures/(图表)
│   └── pdf/(PDF 报告)
└── results/(回测结果)

这个结构覆盖了 80% 的量化项目需求。data 跟 notebooks 是研究阶段主战场,src 是把研究成果固化的代码,reports 是给老板看的输出。

.gitignore 模板 — 防止把不该提交的东西提交到 git,这些必须忽略。

# Python
__pycache__/
*.pyc
*.pyo
.venv/
venv/
.env

# Jupyter
.ipynb_checkpoints/

# 数据(数据通常很大不应该进 git)
data/raw/
data/processed/
*.csv
*.parquet
*.h5
*.pickle

# 模型
models/
*.pkl
*.joblib

# 报告输出
reports/figures/*.png
reports/pdf/*.pdf
results/*.csv

# IDE
.vscode/
.idea/

# 系统
.DS_Store
Thumbs.db

# 敏感配置(API key 之类的)
config/secrets.yml
.env.local

这个 .gitignore 是国内量化机构的标准配置。把它放到项目根目录,git 就会自动忽略这些文件。

注意点 — 数据 / 模型 / 报告输出这三类绝对不要提交 git。数据太大、模型属于「中间结果」、报告输出每次重跑都会变。API key、密码、token 这种敏感信息也绝对不能提交 git,提交之后再删也救不回来(git 历史里能挖出来),只能换 key。


## 实操:量化环境自检 — Python / 包 / 性能 / 中文字体 / 镜像源 全面体检(中国版 — 跟原版相同,纯本地不联网)

下面这段代码用 akshare 抓数据,国内零 VPN 跑通。**直接 Run All** 看结果。

**依赖:** `pip install pandas numpy matplotlib akshare statsmodels scipy`

In [3]:
# day_031_env_check.py — 量化环境完整体检脚本
# 检查:Python 版本 / conda 信息 / 关键包版本 / 性能基准 / 中文字体 / 镜像源
import sys
import os
import platform
import subprocess
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt

print('==== 1. 系统与 Python 信息 ====')
print(f'操作系统:{platform.system()} {platform.release()}')
print(f'机器架构:{platform.machine()}')
print(f'Python 版本:{sys.version.split()[0]}')
print(f'Python 路径:{sys.executable}')

# ============ 2. Conda 环境信息 ============
print('\n==== 2. Conda 环境信息 ====')
try:
    result = subprocess.run(['conda', 'info', '--envs'], capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        envs_lines = result.stdout.strip().split('\n')
        envs = [l for l in envs_lines if l and not l.startswith('#')]
        print(f'Conda 检测到 {len(envs)} 个虚拟环境')
        for env in envs[:5]:
            print(f'  {env}')
    else:
        print('Conda 未安装或未在 PATH 中。建议安装 Miniconda。')
except (FileNotFoundError, subprocess.TimeoutExpired):
    print('Conda 未安装或未在 PATH 中。建议安装 Miniconda。')

current_env = os.environ.get('CONDA_DEFAULT_ENV', os.environ.get('VIRTUAL_ENV', '系统 Python(未隔离)'))
print(f'当前活跃环境:{current_env}')

# ============ 3. 关键包版本检查 ============
print('\n==== 3. 量化核心包版本 ====')
packages = [
    ('numpy', 'NumPy 数值计算'),
    ('pandas', 'Pandas 数据表'),
    ('scipy', 'SciPy 科学计算'),
    ('matplotlib', 'Matplotlib 绘图'),
    ('sklearn', 'Scikit-learn 机器学习'),
    ('yfinance', 'yfinance 海外数据'),
    ('akshare', 'akshare 国内数据'),
    ('statsmodels', 'StatsModels 统计建模'),
    ('jupyter', 'Jupyter 笔记本'),
]

installed, missing = [], []
for pkg, desc in packages:
    try:
        if pkg == 'sklearn':
            import sklearn
            ver = sklearn.__version__
        elif pkg == 'jupyter':
            ver = subprocess.run(['jupyter', '--version'], capture_output=True, text=True, timeout=3).stdout.split('\n')[0] or '已安装'
        else:
            mod = __import__(pkg)
            ver = getattr(mod, '__version__', '未知')
        installed.append((pkg, desc, ver))
        print(f'  ✓ {pkg:13s} {ver:15s} {desc}')
    except (ImportError, FileNotFoundError, subprocess.TimeoutExpired):
        missing.append((pkg, desc))
        print(f'  ✗ {pkg:13s} {"未安装":15s} {desc}')

print(f'\n安装完成 {len(installed)} 个 / 待安装 {len(missing)} 个')
if missing:
    pkg_names = [p for p, _ in missing]
    print(f'建议安装:pip install {" ".join(pkg_names)}')

# ============ 4. NumPy 性能基准 ============
print('\n==== 4. NumPy 性能基准 ====')
N = 1_000_000

# Python 列表版
t0 = time.perf_counter()
py_list = list(range(N))
py_squares = [x * x for x in py_list]
t_list = time.perf_counter() - t0

# NumPy 版
t0 = time.perf_counter()
np_arr = np.arange(N)
np_squares = np_arr * np_arr
t_np = time.perf_counter() - t0

speedup = t_list / t_np if t_np > 0 else 0
print(f'计算 {N:,} 个数的平方:')
print(f'  Python 列表:{t_list*1000:7.1f} ms')
print(f'  NumPy 数组:{t_np*1000:7.1f} ms')
print(f'  NumPy 加速比:{speedup:.0f} 倍')

# ============ 5. 中文字体检查 ============
print('\n==== 5. Matplotlib 中文字体检查 ====')
from matplotlib.font_manager import fontManager
cn_fonts = []
for f in fontManager.ttflist:
    name = f.name
    if any(k in name for k in ['CJK', 'Hei', 'Song', 'Kai', 'Yi', 'Sim', 'WenQuanYi', 'PingFang', 'Noto Sans CJK', 'Noto Serif CJK']):
        if name not in cn_fonts:
            cn_fonts.append(name)
if cn_fonts:
    print(f'检测到 {len(cn_fonts)} 个中文字体:')
    for f in cn_fonts[:5]:
        print(f'  ✓ {f}')
else:
    print('  ✗ 未检测到中文字体!画图标题会显示方块。')
    print('  解决:Windows 系统自带 SimHei,Mac 自带 PingFang,Linux 装 fonts-noto-cjk。')

# ============ 6. 镜像源检查 ============
print('\n==== 6. pip 镜像源检查 ====')
try:
    result = subprocess.run(['pip', 'config', 'list'], capture_output=True, text=True, timeout=5)
    if 'tuna' in result.stdout or 'aliyun' in result.stdout or 'tencent' in result.stdout:
        print('  ✓ 已配置国内镜像源:')
        for line in result.stdout.strip().split('\n'):
            if 'index-url' in line or 'extra-index-url' in line:
                print(f'    {line}')
    else:
        print('  ✗ 未配置国内镜像源,装包会很慢。')
        print('  建议:pip config set global.index-url https://pypi.tuna.tsinghua.edu.cn/simple')
except (FileNotFoundError, subprocess.TimeoutExpired):
    print('  pip 命令未找到')

# ============ 7. 体检总图 ============
print('\n==== 7. 生成体检报告图表 ====')

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# 7-1. 包安装状态
ax = axes[0, 0]
ok_count = len(installed)
miss_count = len(missing)
ax.pie([ok_count, miss_count] if miss_count > 0 else [ok_count],
       labels=[f'已安装 {ok_count}', f'未安装 {miss_count}'] if miss_count > 0 else [f'已安装 {ok_count}'],
       colors=['#4ECDC4', '#FF6B6B'] if miss_count > 0 else ['#4ECDC4'],
       autopct='%1.0f%%', startangle=90)
ax.set_title('① 核心包安装状态', fontsize=13, fontweight='bold')

# 7-2. NumPy 性能对比
ax = axes[0, 1]
ax.bar(['Python 列表', 'NumPy 数组'], [t_list*1000, t_np*1000], color=['#FFE66D', '#4ECDC4'], edgecolor='black')
ax.set_ylabel('耗时 ms', fontsize=11)
ax.set_title(f'② NumPy vs 纯 Python({N:,} 平方)— 加速 {speedup:.0f} 倍', fontsize=12, fontweight='bold')
for i, v in enumerate([t_list*1000, t_np*1000]):
    ax.text(i, v + max(t_list*1000, t_np*1000)*0.02, f'{v:.1f}', ha='center', fontsize=11)
ax.set_yscale('log')
ax.grid(axis='y', alpha=0.3)

# 7-3. 中文字体测试
ax = axes[1, 0]
ax.text(0.5, 0.7, '中文字体测试', ha='center', va='center', fontsize=22, fontweight='bold', transform=ax.transAxes)
ax.text(0.5, 0.4, '茅台 / 沪深 300 / 比特币 / 期权希腊字母', ha='center', va='center', fontsize=14, transform=ax.transAxes)
ax.text(0.5, 0.2, '上涨 / 下跌 / 收益率 / 夏普比率 / 最大回撤', ha='center', va='center', fontsize=14, transform=ax.transAxes)
ax.set_title('③ 中文字体显示测试', fontsize=13, fontweight='bold')
ax.axis('off')

# 7-4. 环境总结
ax = axes[1, 1]
summary = [
    f'操作系统:{platform.system()}',
    f'Python:{sys.version.split()[0]}',
    f'当前环境:{current_env}',
    f'核心包:{ok_count} / {ok_count + miss_count} 已装',
    f'中文字体:{len(cn_fonts)} 个可用',
    f'NumPy 加速比:{speedup:.0f} 倍',
]
ax.text(0.05, 0.95, '量化环境体检总结', fontsize=14, fontweight='bold', transform=ax.transAxes, va='top')
for i, line in enumerate(summary):
    ax.text(0.05, 0.85 - i*0.12, '  • ' + line, fontsize=12, transform=ax.transAxes, va='top')
ax.set_title('④ 体检总结', fontsize=13, fontweight='bold')
ax.axis('off')

plt.tight_layout()
plt.savefig('chart_01.png', dpi=120, bbox_inches='tight')
plt.close()
print('保存 chart_01.png')

# ============ 8. 体检结论 ============
print('\n==== 8. 体检结论 ====')
score = 0
checks = []
if ok_count >= len(packages) - 2: score += 30; checks.append('✓ 核心包齐全 (+30)')
else: checks.append(f'✗ 核心包缺 {miss_count} 个 (建议补装)')
if cn_fonts: score += 20; checks.append('✓ 中文字体可用 (+20)')
else: checks.append('✗ 中文字体缺失')
if speedup >= 50: score += 20; checks.append(f'✓ NumPy 加速比 {speedup:.0f} 倍 (+20)')
elif speedup >= 20: score += 10; checks.append(f'⚠ NumPy 加速比偏低 {speedup:.0f}')
if 'tuna' in (subprocess.run(['pip', 'config', 'list'], capture_output=True, text=True, timeout=3).stdout if True else ''):
    score += 15; checks.append('✓ 国内镜像源已配 (+15)')
else: checks.append('✗ 国内镜像源未配,装包会慢')
if 'conda' in current_env.lower() or 'venv' in current_env.lower() or current_env != '系统 Python(未隔离)':
    score += 15; checks.append('✓ 已使用虚拟环境 (+15)')
else: checks.append('✗ 当前用系统 Python,强烈建议建虚拟环境')

for c in checks:
    print(f'  {c}')
print(f'\n总分:{score} / 100')
if score >= 80:
    print('🎉 优秀 — 你的环境已经是专业研究员水平,直接进 P2 学习。')
elif score >= 60:
    print('⚠ 良好 — 还差几项,补完更顺。')
else:
    print('✗ 不及格 — 按上面 ✗ 项目挨个补,再跑一遍体检。')

==== 1. 系统与 Python 信息 ====
操作系统:Linux 6.6.114.1-microsoft-standard-WSL2
机器架构:x86_64
Python 版本:3.12.3
Python 路径:/mnt/d/huizi_ai_project/ai_course_video/quant365/bin/python

==== 2. Conda 环境信息 ====
Conda 未安装或未在 PATH 中。建议安装 Miniconda。
当前活跃环境:/mnt/d/huizi_ai_project/ai_course_video/quant365

==== 3. 量化核心包版本 ====
  ✓ numpy         2.4.4           NumPy 数值计算
  ✓ pandas        3.0.2           Pandas 数据表
  ✓ scipy         1.17.1          SciPy 科学计算
  ✓ matplotlib    3.10.9          Matplotlib 绘图
  ✓ sklearn       1.8.0           Scikit-learn 机器学习
  ✓ yfinance      1.3.0           yfinance 海外数据
  ✓ akshare       1.18.60         akshare 国内数据
  ✓ statsmodels   0.14.6          StatsModels 统计建模
  ✗ jupyter       未安装             Jupyter 笔记本

安装完成 8 个 / 待安装 1 个
建议安装:pip install jupyter

==== 4. NumPy 性能基准 ====
计算 1,000,000 个数的平方:
  Python 列表:   39.0 ms
  NumPy 数组:    6.0 ms
  NumPy 加速比:6 倍

==== 5. Matplotlib 中文字体检查 ====
检测到 2 个中文字体:
  ✓ Noto Sans CJK JP
  ✓ Noto Serif CJK JP

==== 6. pip 镜像源检查 ====


## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| 私募研究员入职 SOP | 某量化私募内部培训手册 | 某头部量化私募的入职培训手册第一周第一天就是「环境搭建」,流程跟我们今天讲的几乎一模一样 — Miniconda 装好、conda create 建项目环境、conda install + pip install 装包、配清华镜像源、装 VS Code + Jupyter Lab。这套流程要求新员工 4 小时内全部走完,跑通一个「拉数据 + 画图」demo 项目。能跑通才算过第一关,过不了第一关需要重学。这套标准化流程让该公司新员工平均 2 周内能独立做项目,大幅缩短了上手时间。 |
| GitHub 量化项目 Cookiecutter 模板 | cookiecutter-data-science | DrivenData 团队开源的 cookiecutter-data-science 项目(GitHub 8000+ star)就是一个标准的量化项目目录结构模板。装 cookiecutter 之后一行命令就能生成完整目录结构,直接开搞。这个模板是全球数据科学社区的事实标准,国内不少量化机构也直接用或者基于它改。今天我们讲的目录结构基本就是这个模板的简化版。建议过几个月后你尝试用这个 cookiecutter 自动化生成项目骨架。 |
| 国内清华 / 北大 / 上交 量化课作业要求 | 高校量化课程作业评分标准 | 国内顶级高校量化金融课程,期末项目评分标准里专门有一项叫「工程化」,占总分大约 20-30%。具体要求包括 — 必须用虚拟环境 + requirements.txt(扣 5 分如果没用)、必须用标准目录结构(扣 5 分如果一堆文件丢在根目录)、必须有 README + 复现指令(扣 5 分如果别人按你的代码跑不起来)、必须有单元测试(扣 5 分如果没有)。这种严格要求倒逼学生养成工程化习惯,毕业后就业明显比其他学校快。今天讲的内容是这种工程化的最低门槛。 |
| WorldQuant Brain 量化平台 | 全球 Alpha 众包平台 | WorldQuant 旗下的 Brain 平台,全球用户可以用平台提供的数据接口编写 alpha 策略,通过审核后获得真金白银的提成。平台对提交策略的代码格式 / 命名规则 / 文档规范要求非常严格,任何不符合规范的提交直接打回。这种「严格规范换提成」的机制,逼着每一个全球量化爱好者养成工程化习惯。WorldQuant Brain 的代码规范跟我们今天讲的目录结构 + .gitignore 思路高度一致。 |
| 国内某私募技术故障复盘 | 2023 年某私募一夜回退事件 | 2023 年国内某私募,一位研究员把自己机器上的代码部署到生产服务器,因为没用 requirements.txt 锁定包版本,生产服务器上自动装了最新版的 pandas 2.0,而该研究员开发用的 pandas 1.5。某些 API 行为发生了 breaking change,生产代码跑出来的信号跟回测完全不一样,当天亏了 800 万。事后复盘的根本原因就是「没有锁定依赖版本」。该公司从那以后强制要求所有部署项目必须用 requirements.txt + pip-tools 锁定到精确版本号,任何升级必须经过 PR 审核。这就是不工程化的真实代价。 |


## 常见坑

### ⚠ 01. 直接装包到系统 Python

新手最常见错误 — 不用虚拟环境,直接 pip install 装到系统 Python。短期没问题,长期版本冲突频繁、卸载困难、跨项目互相干扰。**正确做法**:任何项目都用虚拟环境,系统 Python 保持干净不动。一台电脑可以有几十个虚拟环境,但系统 Python 只有一个。

### ⚠ 02. conda 和 pip 混用引发幽灵 bug

在同一个环境里 conda install 装一些、pip install 装一些,有时候同一个包两边都装了不同版本,引发非常难调的 bug。**正确做法**:同一个环境里有先后顺序 — 优先 conda 装,conda 没有的再用 pip 装,装完之后不要再用 conda 升级 / 改动该包。或者直接全用 pip,不混用。

### ⚠ 03. 环境共享但没冻结版本

做研究时跑得通的代码,分享给同事或部署到服务器跑不起来,通常是因为 requirements.txt 没写明确版本号,或者根本没生成。**正确做法**:每个项目都要有 requirements.txt 或 environment.yml,版本号必须冻结到 minor(如 pandas==2.0.3),pin 到具体到 patch 版本更安全。生成命令 pip freeze > requirements.txt 或者 conda env export > environment.yml。

### ⚠ 04. API key / 密码提交进了 git

Tushare token、券商 API key、邮件密码这类敏感信息,一旦提交进 git 就再也删不干净 — git 历史里能挖出来。这种事故每年都有新闻,被黑客扫到立刻被恶意调用。**正确做法**:敏感信息放 .env 文件或 config/secrets.yml,这些文件加进 .gitignore 永远不提交。代码里用 os.getenv 读取。事故发生后立刻换 key,千 万别想着「我删一个 commit 就行」。

### ⚠ 05. 在系统盘装一堆环境塞爆 C 盘

Windows 用户最常见 — Anaconda 默认装到 C:\Users\用户名\anaconda3,每个环境占 几个 GB,装多几个环境 C 盘就满了。Mac/Linux 用户也有类似问题。**正确做法**:Anaconda 安装时选择装到 D 盘 / E 盘 / 数据盘。如果已经装到 C 盘了,可以用 conda config --add envs_dirs D:\conda_envs 改默认环境路径。这一招省下大量系统盘空间。

## 实战 SOP · 量化环境搭建 7 条 SOP

1. 新手第一次装 Anaconda 熟悉一下,用过 1 个月之后切换到 Miniconda 保持环境干净
2. 每个项目独立虚拟环境 — conda create -n 项目名 python=3.11,绝不在系统 Python 装包
3. 配国内镜像源 — pip 用清华源、conda 也用清华源,装包速度提升 10 倍
4. Jupyter Lab 用于研究探索,VS Code 用于写库调试部署,两者组合不要二选一
5. 每个项目根目录有 README + requirements.txt + .gitignore 三件套
6. 敏感信息(API key / 密码 / token)绝不提交 git,放 .env 文件 + .gitignore
7. 数据 / 模型 / 报告输出绝不提交 git,文件太大 + 中间结果 + 容易污染历史

> 把这段打印贴在你电脑边。

## 总结 · 你应该带走的

2. 虚拟环境的核心价值 — 隔离 + 可复现 + 清理方便,任何超过 100 行的项目都必须用
3. Anaconda 适合新手第一次入门,Miniconda 适合熟手,pip + venv 是官方原生方案,poetry 适合做开源库
4. 量化必装 15 个包清单 — numpy / pandas / scipy / statsmodels / sklearn / matplotlib / seaborn / plotly / yfinance / akshare / quantstats / empyrical / ffn / pyfolio / jupyter
5. Jupyter Lab(研究 / 探索 / 出报告)+ VS Code(写库 / 调试 / 部署)双轨制,各自适合不同场景
6. 标准量化项目目录 — data / notebooks / src / tests / reports / results,加 README + requirements + .gitignore 三件套
7. 国内镜像源配置 — pip 用清华源,conda 也用清华源,装包速度提升 10 倍,新手必配
8. 敏感信息 + 数据 + 模型 + 报告输出绝不提交 git,有专门的 .gitignore 模板可以直接抄

## 自测题

**Q1.** 为什么必须用虚拟环境而不是直接装到系统 Python?给出至少 3 个具体的工程理由。(提示:版本冲突 + 可复现性 + 清理方便)

**Q2.** Anaconda / Miniconda / pip / poetry 这 4 个工具,量化场景里推荐用哪个?为什么?(提示:Miniconda 主战场 + 偶尔 pip 补缺,平衡轻量跟功能)

**Q3.** Jupyter Lab 和 VS Code 各自适合什么场景?为什么推荐双轨制?(提示:研究探索 vs 写库部署,工作流不同)

**Q4.** .gitignore 里至少应该忽略哪几类文件?为什么?(提示:虚拟环境 + 数据 + 模型 + 敏感信息 + IDE 配置 + 系统文件)

**Q5.** 如果你的 pandas 在 A 项目用 1.5,在 B 项目用 2.0,会发生什么问题?怎么解决?(提示:版本冲突;每个项目独立 conda 环境)

把答案写下来,3 天后再回看。

## 下一节预告

**Day 032 · NumPy 入门** (NumPy Basics)

第 32 课 NumPy 入门。环境搭好了,我们正式开始学量化最底层的工具栈。NumPy 是所有 Python 数据科学库的基石,pandas / sklearn / matplotlib 内部都用 NumPy。这一节讲清楚 ndarray 是什么、广播怎么运作、矢量化为什么比 for 循环快 100 倍、视图跟拷贝什么区别、5 个最常用 API 怎么用。最后用 NumPy 写一个最经典的金融指标 — 移动平均,让你看到「向量化思维」的力量。

## 推荐阅读

- Conda 官方文档《User Guide》(免费在线,docs.conda.io)— Conda 的权威使用手册,装包 / 建环境 / 配置镜像全在这
- Wes McKinney《Python for Data Analysis》(2022 第 3 版,O'Reilly)— 第 1 章详细讲数据科学环境搭建,Pandas 作者亲自写,业内标准入门书
- DrivenData《Cookiecutter Data Science》(GitHub 开源,8000+ star)— 数据科学项目目录结构事实标准,可以直接用模板生成项目骨架
- Jupyter Project《Jupyter Lab Documentation》(免费在线,jupyterlab.readthedocs.io)— Jupyter Lab 的完整文档,从基本用法到插件开发全覆盖
- Python 工具栈进阶 — pip-tools(自动生成精确依赖锁定文件)、pre-commit(Git 提交前自动跑检查)、direnv(自动激活虚拟环境)、mamba(conda 的 10 倍速版本,装包飞快);这四个工具是工程化二级进阶,P2 学完可以尝试